In [1]:
import torch
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from torch.utils.data import DataLoader
from torch import nn
import numpy as np
import math
from scipy.spatial.distance import cosine

# Task


-- Workout that a user has done = positive workout

-- Workout that a user has not done = negative feedback

In [2]:
# Read data
df = pd.read_csv('../data/main_derived.csv')

In [3]:
# Create user profile
user_profiles = pd.DataFrame(
    df.groupby('userId', as_index=False).agg({
        'sport': lambda x: set(x),
        'gender': 'first',
    })
)

# all unique workout types  in this dataset
all_sports = set(df['sport'].unique())

# workout that this user has not done before
user_profiles['new_sport'] = user_profiles['sport'].apply(lambda played_sport: all_sports - played_sport)

user_profiles

,userId,sport,gender,new_sport
0,69,"{bike, run, mountain bike}",male,"{circuit training, soccer, cross-country skiin..."
1,2358,"{bike, circuit training, weight training, run,...",male,"{soccer, cross-country skiing, kite surfing, b..."
2,3808,"{bike, run, walk}",male,"{circuit training, soccer, cross-country skiin..."
3,4101,"{bike, run, mountain bike}",male,"{circuit training, soccer, cross-country skiin..."
4,4434,"{bike, run, bike (transport)}",male,"{circuit training, soccer, cross-country skiin..."
...,...,...,...,...
1051,14671923,"{bike, run}",male,"{circuit training, soccer, cross-country skiin..."
1052,14779650,"{bike, run}",male,"{circuit training, soccer, cross-country skiin..."
1053,15279967,{run},male,"{circuit training, soccer, cross-country skiin..."
1054,15326644,{run},male,"{circuit training, soccer, cross-country skiin..."


In [4]:
# workout profile
workout_profiles = df.groupby('sport', as_index=False).agg({
    'distance': 'mean',
    'duration': 'mean',
    'heart_rate': lambda x: math.ceil(np.mean(x)),
    'calories': 'mean'
})

workout_profiles['calories_per_minute'] = workout_profiles['calories'] / 60
workout_profiles

,sport,distance,duration,heart_rate,calories,calories_per_minute
0,aerobics,0.798665,3830.285714,146,556.347143,9.272452
1,badminton,0.864421,4026.200000,141,819.330000,13.655500
2,basketball,2.542059,4387.500000,127,871.986125,14.533102
3,bike,25.491494,6476.882977,134,998.768059,16.646134
4,bike (transport),13.002559,3621.162764,127,629.012441,10.483541
5,circuit training,1.339635,3854.235955,121,552.574652,9.209578
6,climbing,2.132856,8867.200000,135,1556.174000,25.936233
7,core stability training,7.956175,4660.517730,130,696.467645,11.607794
8,cross-country skiing,11.237095,6422.420584,138,1132.937025,18.882284
9,downhill skiing,13.751077,9102.808511,85,876.484617,14.608077


# Model Formulation

In [5]:
class FitRecDataset():
    def __init__(self, data_path):
        # Get an encoder
        df = pd.read_csv(data_path)

        # Convert numerical values to int
        numerical_columns = df.select_dtypes(include=['number']).columns 
        for col in numerical_columns:
            df[col] = df[col].apply(lambda x: math.floor(x))

        # Re-order columns 
        df = df[['userId', 'gender', 'duration', 'heart_rate', 'distance','calories', 'sport']]

        # Get column names
        col_names = df.columns
        
        # Get an encoder
        self.encoder = OrdinalEncoder()

        df = self.encoder.fit_transform(df)

        self.features = torch.tensor(df[:, :-1], dtype=int)
        self.labels = torch.tensor(df[:, -1], dtype=int)
        self.features_dim = {}

        for feature_name, cat in zip(col_names, self.encoder.categories_):
            self.features_dim[feature_name] = len(cat)
            
    def __len__(self):
        return len(self.features)

    def get_encoder(self):
        return self.encoder

    def __getitem__(self, idx):
        """
        Return a row of data.
        """
        
        return self.features[idx], self.labels[idx]


In [6]:
# Load data
data_path = '../data/main_derived.csv'
dataset = FitRecDataset(data_path)

In [7]:
# Sample data getter
dataset[0]

(tensor([   43,     1, 10291,   102,    56,  3639]), tensor(3))

In [8]:
class UserModel(nn.Module):
    def __init__(self, user_dim, gender_dim, output_dim):
        """
        A user is defined by
            userId
            gender
        """
        super().__init__()
        self.id_embedding = nn.Embedding(user_dim, output_dim)
        self.gender_embedding = nn.Embedding(gender_dim, output_dim)
        
    def forward(self, input_features):
        # Get featuers from the input
        user_ids = input_features[:, 0]
        genders = input_features[:, 1]

        # Transform into embedding dimension
        id_embeddings = self.id_embedding(user_ids)
        gender_embeddings = self.gender_embedding(genders)

        # Concatenate features
        user_embedidng = torch.cat((id_embeddings, gender_embeddings), dim=1)
        return user_embedidng

In [9]:
class WorkoutModel(nn.Module):
    """
    A workout model is defined by:
        duration
        heart_rate
        distance
        calories
    """
    def __init__(self, duration_dim, hr_dim, distance_dim, calories_dim, output_dim):
        super().__init__()
        # embedding layers
        self.duration_embedding = nn.Embedding(duration_dim, output_dim)
        self.hr_embedding = nn.Embedding(hr_dim, output_dim)
        self.distance_embedding = nn.Embedding(distance_dim, output_dim)
        self.cal_embedding = nn.Embedding(calories_dim, output_dim)

        
        self.fc1 = nn.Linear(2, output_dim)
        self.fc2 = nn.Linear(output_dim, output_dim)
        self.relu = nn.ReLU()
        
    def forward(self, input_features):
        durations = input_features[:, 2]
        heart_rates = input_features[:, 3]
        distances = input_features[:, 4]
        calories = input_features[:, 5]

        dur_embedding = self.duration_embedding(durations)
        dis_embedding = self.distance_embedding(distances)
        cal_embedding = self.cal_embedding(calories)
        hr_embedding = self.hr_embedding(heart_rates)

        workout_embedding = torch.cat((dur_embedding, dis_embedding, cal_embedding, hr_embedding), dim=1)

        return workout_embedding

In [18]:
class FitRecModel(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        duration_dim = dataset.features_dim['duration']
        distance_dim = dataset.features_dim['distance']
        gender_dim = dataset.features_dim['gender']
        calories_dim = dataset.features_dim['calories']
        hr_dim = dataset.features_dim['heart_rate']
        user_dim = dataset.features_dim['userId']
        
        self.workout_model = WorkoutModel(duration_dim, hr_dim, distance_dim, calories_dim, output_dim)
        self.user_model =  UserModel(user_dim, gender_dim, output_dim=2*output_dim)

        self.fc1 = nn.Linear(128,16) 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, dataset.features_dim['sport']) 

    def forward(self, train_features):
        w_rep = self.workout_model(train_features)
        u_rep = self.user_model(train_features)

        result = (w_rep @ u_rep.T)

        x = self.relu(self.fc1(result))
        x = self.fc2(x)
        
        return x

In [19]:
# batch training and testing data
train_dataloader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)
test_dataloader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

In [21]:
# Define model
model = FitRecModel(output_dim=30)

# Loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


running_loss = 0
epochs = 11

# Train for 5 epochs
for epoch in range(1, epochs):
    epoch_loss = 0
    for i, (train_features, train_labels) in enumerate(train_dataloader):
        optimizer.zero_grad()
    
        outputs = model(train_features)
    
        loss = loss_fn(outputs, train_labels)
        epoch_loss += loss.item()

        loss.backward()
        optimizer.step()

    correct = 0
    total = 0
    model.eval()
    for train_features, train_labels in train_dataloader:
        outputs = model(train_features)
        preds = outputs.softmax(dim=1).argmax(dim=1)
        correct += (preds == train_labels).sum().item()
        total += len(preds)
    
    print(f'Epoch {epoch} -- Loss: {epoch_loss / len(train_dataloader):5} // Acc: {correct / total}')

Epoch 1 -- Loss: 1.2503438170083045 // Acc: 0.7314174537393986
Epoch 2 -- Loss: 0.8902504586585228 // Acc: 0.7823221858134156
Epoch 3 -- Loss: 0.822113313096803 // Acc: 0.7979652563608327
Epoch 4 -- Loss: 0.7928384785341499 // Acc: 0.804380300693909
Epoch 5 -- Loss: 0.7697089071814244 // Acc: 0.8080486218195837
Epoch 6 -- Loss: 0.752132608441086 // Acc: 0.8120120952197378
Epoch 7 -- Loss: 0.7359238344610151 // Acc: 0.8136324209714726
Epoch 8 -- Loss: 0.722837505840575 // Acc: 0.8169513781804163
Epoch 9 -- Loss: 0.711737873651656 // Acc: 0.8169453546646106
Epoch 10 -- Loss: 0.7018888774245357 // Acc: 0.8187042212798766


In [22]:
# Get trained embedding models
workout_model = model.workout_model
user_model = model.user_model

# Get all workout embeddings
workouts = []
features = []
for train_features, train_labels in train_dataloader:
    for i, row in enumerate(train_features):
        workout_embedding = workout_model(row.reshape(1, -1))
        workouts.append(workout_embedding)
        features.append(torch.cat((row, train_labels[i].reshape(1)), dim=0))

# Read dataframe for recommendations
df = pd.read_csv('../data/main_derived.csv')

In [23]:
def get_recommendation(user_id, gender):
    user_embedding = user_model(torch.tensor([user_id,gender]).reshape(1,-1))
    scores = []

    for workout in workouts:
        sim_score = 1  - cosine(user_embedding.detach().numpy().flatten(), workout.detach().numpy().flatten())
        scores.append(sim_score)

    # Indicies of most similar workouts
    indices = np.argpartition(scores, -5)[-5:]

    # Most similar workouts using above indicies
    most_similar = np.array(scores)[indices]

    # Inverse transform features of most similar workouts
    top_features = dataset.encoder.inverse_transform(np.array(features)[indices])

    # Inverse transform to get original user ID
    original_user_id = dataset.encoder.inverse_transform(
        torch.tensor(
            [user_id, gender, 1, 1, 1, 1, 1] # Dummy values except the first two columns
        ).reshape(1, -1))[0][0]
    
    print(f'You are {get_user_info(original_user_id)}')        
    print(f"My recommendations: \n")

    for row in top_features:
        recommended_uid, gender, duration, heart_rate, distance, calories, sports = row
        print(get_user_info(recommended_uid))
        

def get_user_info(user_id):
    stats = df[df['userId'] == user_id][['duration', 'heart_rate', 'distance', 'calories']].mean()
    sports_played = df[df['userId'] == user_id].sport.unique()
    return f"""👤 User {user_id}:
    ⏱️ Average workout duration: \t\t{stats['duration']}
    ❤️ Average heart rate: \t\t\t{stats['heart_rate']}
    🛣️ Average distance cover: \t\t\t{stats['distance']}
    🔥 Average calories burn: \t\t\t{stats['calories']}
    
    🏋️ Workout history: \t\t\t{', '.join(sports_played)[:50]}...
    """

In [24]:
get_recommendation(123, 1)

You are 👤 User 757635:
    ⏱️ Average workout duration: 		6883.14
    ❤️ Average heart rate: 			140.64896
    🛣️ Average distance cover: 			30.94268519200521
    🔥 Average calories burn: 			1178.5586

    🏋️ Workout history: 			bike, mountain bike...
    
My recommendations: 

👤 User 104216:
    ⏱️ Average workout duration: 		3990.6451612903224
    ❤️ Average heart rate: 			153.24883870967741
    🛣️ Average distance cover: 			6.228592209779347
    🔥 Average calories burn: 			773.5806451612904

    🏋️ Workout history: 			run, bike...
    
👤 User 5850177:
    ⏱️ Average workout duration: 		5406.825
    ❤️ Average heart rate: 			127.42575000000002
    🛣️ Average distance cover: 			6.619606020528536
    🔥 Average calories burn: 			977.75

    🏋️ Workout history: 			run...
    
👤 User 2587574:
    ⏱️ Average workout duration: 		4943.330188679245
    ❤️ Average heart rate: 			137.33972641509436
    🛣️ Average distance cover: 			15.211219266168367
    🔥 Average calories burn: 			814.190089622